In [19]:
"""
Heterogeneous Fire Effects: Size, Severity, Vegetation, and Recreation Counties
===============================================================================

Estimates DiD fire effects stratified by fire size, severity, vegetation, and recreation counties
using Sun-Abraham saturated event study.
"""

import pandas as pd
import numpy as np
import re
import warnings
from pathlib import Path

warnings.filterwarnings('ignore', message='.*variables dropped due to multicollinearity.*')
warnings.filterwarnings('ignore', module='pyfixest')

# ============================================================
# PATHS (relative to this notebook)
# ============================================================
try:
    HERE = Path(__file__).resolve().parent
except NameError:
    HERE = Path.cwd()

DATA_DIR = (HERE / "../data").resolve()
OUTPUT_DIR = (HERE / "../outputs/dif_in_dif").resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# YOUR DATA FILES
# ============================================================

# Colorado
CO_PRED = (DATA_DIR / "03_visitation/TreatCon_Pred_CO.csv").resolve()
CO_FIRE = (DATA_DIR / "02_site_features/fire_feature_matrix_CO.csv").resolve()

# California
CA_PRED = (DATA_DIR / "03_visitation/TreatCon_Pred_CA.csv").resolve()
CA_FIRE = (DATA_DIR / "02_site_features/fire_feature_matrix_CA.csv").resolve()

# Recreation county mappings (preprocessed CSV - no shapefile needed!)
RECREATION_MAPPINGS_CSV = (DATA_DIR / "02_site_features/siteid_recreation_status.csv").resolve()

# Where to save results
OUTPUT = (OUTPUT_DIR / "did_heterogeneity_results.csv").resolve()

# ============================================================
# ANALYSIS SETTINGS
# ============================================================

MONTHS_PRE_FIRE = 12
REFERENCE_MONTH = -2
MIN_SAMPLE_PERCENT = 0.10

# Control variables
CONTROLS = ['temperature_mean']

# ============================================================
# LOAD RECREATION COUNTY MAPPINGS (FROM CSV)
# ============================================================

print("=" * 60)
print("LOADING RECREATION COUNTY DATA")
print("=" * 60)

def load_recreation_mappings():
    """Load recreation mappings from preprocessed CSV"""
    
    print(f"\nLoading recreation mappings from: {RECREATION_MAPPINGS_CSV.name}")
    rec_df = pd.read_csv(RECREATION_MAPPINGS_CSV)
    
    print(f"  ✓ Loaded {len(rec_df)} siteid mappings")
    
    # Create state-specific dictionaries
    co_rec = rec_df[rec_df['state'] == 'CO'].set_index('siteid')['recreation_status'].to_dict()
    ca_rec = rec_df[rec_df['state'] == 'CA'].set_index('siteid')['recreation_status'].to_dict()
    
    # Add case-insensitive versions
    co_rec_full = {}
    for sid, status in co_rec.items():
        co_rec_full[sid] = status
        co_rec_full[sid.upper()] = status
        co_rec_full[sid.lower()] = status
    
    ca_rec_full = {}
    for sid, status in ca_rec.items():
        ca_rec_full[sid] = status
        ca_rec_full[sid.upper()] = status
        ca_rec_full[sid.lower()] = status
    
    print(f"  CA siteid mappings: {len(ca_rec)} unique sites")
    print(f"  CO siteid mappings: {len(co_rec)} unique sites")
    
    # Print distribution
    print("\n  Recreation status distribution:")
    print(rec_df.groupby(['state', 'recreation_status']).size())
    
    return co_rec_full, ca_rec_full

# Load recreation mappings
co_rec_mapping, ca_rec_mapping = load_recreation_mappings()

# ============================================================
# CLASSIFICATION FUNCTIONS
# ============================================================

def classify_treatment_co(treatment_type):
    """Colorado treatment classification"""
    if pd.isna(treatment_type):
        return 'unknown'
    tt = str(treatment_type).lower()
    if 'fire' in tt and 'control' not in tt:
        return 'wildfire_treated'
    elif 'controlfire' in tt:
        return 'wildfire_control'
    elif 'rx' in tt and 'control' not in tt:
        return 'rx_treated'
    elif 'controlrx' in tt:
        return 'rx_control'
    return 'unknown'

def classify_treatment_ca(treatment_type):
    """California treatment classification"""
    if pd.isna(treatment_type):
        return "unknown"
    tt = str(treatment_type).lower()
    if "wildfire" in tt and "treated" in tt:
        return "wildfire_treated"
    elif "wildfire" in tt and "control" in tt:
        return "wildfire_control"
    elif "rx" in tt and "treated" in tt:
        return "rx_treated"
    elif "rx" in tt and "control" in tt:
        return "rx_control"
    return "unknown"

def classify_vegetation(row):
    """Classify dominant vegetation type"""
    def safe_float(val):
        if val is None or pd.isna(val):
            return 0.0
        try:
            return float(val)
        except:
            return 0.0
    
    grass = safe_float(row.get('grass_pct_mean', 0))
    shrub = safe_float(row.get('shrub_pct_mean', 0))
    tree = safe_float(row.get('tree_pct_mean', 0))
    
    if grass == 0 and shrub == 0 and tree == 0:
        return 'unknown'
    
    veg_dict = {'grass': grass, 'shrub': shrub, 'tree': tree}
    return max(veg_dict, key=veg_dict.get)

def classify_severity(row, veg_type=None):
    """
    Classify fire severity
    Grass fires auto-classified as low (CBI not ecologically meaningful for grass)
    """
    if veg_type == 'grass':
        return 'low'
    
    def safe_float(val):
        if val is None or pd.isna(val):
            return 0.0
        try:
            return float(val)
        except:
            return 0.0
    
    pct_high = safe_float(row.get('pct_high_severity', 0))
    pct_mod = safe_float(row.get('pct_moderate_severity', 0))
    
    if pct_high > 0.33:
        return 'high'
    if (pct_mod + pct_high) > 0.33:
        return 'moderate'
    return 'low'

def calculate_size_terciles(df):
    """Calculate fire size terciles"""
    treated = df[df['treated'] == 1].copy()
    sizes = treated[treated['area_km2'].notna()]['area_km2'].values
    if len(sizes) < 3:
        return None
    return np.percentile(sizes, [33.33, 66.67])

def classify_fire_size(size, breaks):
    """Classify fire size based on terciles"""
    if breaks is None or pd.isna(size):
        return 'unknown'
    size = float(size)
    if size <= breaks[0]:
        return 'small'
    elif size <= breaks[1]:
        return 'medium'
    return 'large'

# ============================================================
# DiD ESTIMATION (SUN-ABRAHAM)
# ============================================================

def estimate_fire_effect(df, fire_type, controls, severity=None, vegetation=None, 
                        size_class=None, recreation_status=None):
    """
    Estimate monthly DiD effects with optional stratification using Sun-Abraham
    
    Stratification: treated group filtered, control group stays full pool
    """
    try:
        import pyfixest as pf
    except ImportError:
        raise ImportError("Install pyfixest: pip install pyfixest")
    
    # Filter by fire type
    if fire_type == 'wildfire':
        panel = df[df['family'].isin(['wildfire_treated', 'wildfire_control'])].copy()
    else:
        panel = df[df['family'].isin(['rx_treated', 'rx_control'])].copy()
    
    # Get full control pool BEFORE stratification
    full_control = panel[panel['treated'] == 0].copy()
    
    # Apply stratification to TREATED ONLY
    if recreation_status:
        treated = panel[(panel['treated'] == 1) & (panel['recreation_status'] == recreation_status)]
        panel = pd.concat([treated, full_control], ignore_index=True)
    elif size_class:
        treated = panel[(panel['treated'] == 1) & (panel['size_class'] == size_class)]
        panel = pd.concat([treated, full_control], ignore_index=True)
    elif severity:
        treated = panel[(panel['treated'] == 1) & (panel['severity_class'] == severity)]
        panel = pd.concat([treated, full_control], ignore_index=True)
    elif vegetation:
        treated = panel[(panel['treated'] == 1) & (panel['veg_type'] == vegetation)]
        panel = pd.concat([treated, full_control], ignore_index=True)
    
    # Time window
    panel = panel[panel['rel_month'].notna()].copy()
    panel = panel[panel['rel_month'] >= -MONTHS_PRE_FIRE].copy()
    panel['rel_month'] = panel['rel_month'].astype(int)
    
    n_treated = panel[panel['treated'] == 1]['siteid'].nunique()
    n_control = panel[panel['treated'] == 0]['siteid'].nunique()
    
    if n_treated < 2 or n_control < 1:
        return None, None, None, None
    
    # Baseline mean
    baseline = panel[(panel['treated'] == 1) & (panel['rel_month'] < 0)]['outcome'].mean()
    if pd.isna(baseline) or baseline == 0:
        baseline = panel['outcome'].mean()
    
    # Sun-Abraham Preparation
    panel['year_month'] = panel['year'] * 12 + panel['month']
    
    # Create cohort variable (first treatment time for treated units)
    treated_sites = panel.loc[panel['treated'] == 1, 'siteid'].unique()
    cohort_dict = {}
    
    for sid in treated_sites:
        site_data = panel[panel['siteid'] == sid]
        treatment_time = site_data.loc[site_data['rel_month'] == 0, 'year_month']
        if len(treatment_time) > 0:
            cohort_dict[sid] = int(treatment_time.iloc[0])
    
    panel['cohort'] = panel['siteid'].map(cohort_dict)
    
    # Never-treated units: cohort must be NaN (not 0!)
    panel.loc[panel['treated'] == 0, 'cohort'] = np.nan
    
    # Drop treated units without cohort assignment
    panel = panel[~((panel['treated'] == 1) & (panel['cohort'].isna()))].copy()
    
    if len(panel) == 0:
        return None, None, None, None
    
    # Track n_treated per month for min-sample filtering
    n_treated_per_month = panel[panel['treated'] == 1].groupby('rel_month')['siteid'].nunique().to_dict()
    
    # Sun-Abraham Estimation
    available_controls = [c for c in controls if c in panel.columns]
    xfml = " + ".join(available_controls) if available_controls else None
    
    try:
        # Run saturated event study
        fit = pf.event_study(
            data=panel,
            yname='outcome',
            idname='siteid',
            tname='year_month',
            gname='cohort',
            xfml=xfml,
            estimator='saturated'
        )
        
        # Aggregate with share-weighting
        agg = fit.aggregate(weighting='shares')
        
        # Extract monthly effects
        effects = {}
        for rel_time, row in agg.iterrows():
            month = int(round(float(rel_time)))
            coef = float(row['Estimate'])
            se = float(row['Std. Error'])
            
            effects[month] = {
                'coef': coef,
                'se': se,
                'ci_lower': coef - 1.96 * se,
                'ci_upper': coef + 1.96 * se,
                'n_treated': n_treated_per_month.get(month, 0)
            }
        
        # Add reference month (set to zero)
        if REFERENCE_MONTH not in effects:
            effects[REFERENCE_MONTH] = {
                'coef': 0.0, 'se': 0.0, 'ci_lower': 0.0, 'ci_upper': 0.0,
                'n_treated': n_treated_per_month.get(REFERENCE_MONTH, 0)
            }
        
        return dict(sorted(effects.items())), n_treated, n_control, baseline
        
    except Exception as e:
        print(f"Sun-Abraham estimation failed: {str(e)}")
        return None, None, None, None

def aggregate_to_years(monthly_effects):
    """Aggregate monthly to annual using inverse-variance weighting"""
    if not monthly_effects:
        return None
    
    df = pd.DataFrame.from_dict(monthly_effects, orient='index')
    
    # Find minimum sample threshold
    peak_sample = df['n_treated'].max()
    min_sample = max(1, int(np.floor(peak_sample * MIN_SAMPLE_PERCENT)))
    
    # Keep post-fire months with sufficient sample
    df = df[df.index >= 0]
    df = df[df['n_treated'] >= min_sample]
    
    if len(df) == 0:
        return None
    
    # Assign to years
    df['year'] = np.floor(df.index / 12).astype(int) + 1
    df.loc[df['year'] > 5, 'year'] = 5
    
    # Aggregate by year
    annual = []
    for year, group in df.groupby('year'):
        weights = 1 / (group['se']**2 + 1e-10)
        weights = weights / weights.sum()
        
        coef = (group['coef'] * weights).sum()
        se = 1 / np.sqrt((1 / (group['se']**2 + 1e-10)).sum())
        
        annual.append({
            'year': int(year),
            'coef': coef,
            'se': se,
            'ci_lower': coef - 1.96 * se,
            'ci_upper': coef + 1.96 * se
        })
    
    # Add year 0
    annual.insert(0, {
        'year': 0, 'coef': 0.0, 'se': 0.0, 'ci_lower': 0.0, 'ci_upper': 0.0
    })
    
    return {row['year']: row for row in annual}

def to_percent(effects, baseline):
    """Convert to percentage changes"""
    if baseline == 0:
        return effects
    return {
        year: {
            'coef': (vals['coef'] / baseline) * 100,
            'se': (vals['se'] / baseline) * 100,
            'ci_lower': (vals['ci_lower'] / baseline) * 100,
            'ci_upper': (vals['ci_upper'] / baseline) * 100
        }
        for year, vals in effects.items()
    }

# ============================================================
# LOAD DATA
# ============================================================

print("\n" + "=" * 60)
print("LOADING DATA")
print("=" * 60)

def load_state_data(pred_file, fire_file, classifier, state, rec_mapping):
    """Load and prepare data"""
    # Load files
    pred = pd.read_csv(pred_file)
    pred.columns = pred.columns.str.strip().str.lower()
    
    fire = pd.read_csv(fire_file)
    fire.columns = fire.columns.str.strip().str.lower()
    
    # Prepare merge columns
    veg_cols = ['grass_pct_mean', 'shrub_pct_mean', 'tree_pct_mean']
    merge_cols = ['siteid', 'year', 'month']
    fire_merge_cols = merge_cols + CONTROLS + veg_cols + ['area_km2', 'pct_high_severity', 'pct_moderate_severity']
    fire_merge_cols = [c for c in fire_merge_cols if c in fire.columns]
    
    # Merge
    df = pred.merge(
        fire[fire_merge_cols].drop_duplicates(subset=merge_cols),
        on=merge_cols, how='left'
    )
    
    # Remove duplicates
    df = df.loc[:, ~df.columns.duplicated()]
    
    # Classifications
    df['family'] = df['treatment_type'].apply(classifier)
    df['treated'] = df['family'].str.contains('treated').astype(int)
    df['outcome'] = df['y_pred']
    df['time_period'] = df['year'].astype(str) + '_' + df['month'].astype(str).str.zfill(2)
    
    # Fill missing controls
    for col in CONTROLS:
        if col in df.columns:
            df[col] = df[col].fillna(df[col].median())
    
    # Recreation status mapping
    df['siteid_str'] = df['siteid'].astype(str).str.strip()
    df['recreation_status'] = df['siteid_str'].map(rec_mapping)
    
    matched = df['recreation_status'].notna().sum()
    total = len(df)
    print(f"  Recreation status matched: {matched:,}/{total:,} ({100*matched/total:.1f}%)")
    
    # Vegetation classification
    if all(c in df.columns for c in veg_cols):
        veg_by_site = df.groupby('siteid')[veg_cols].first().reset_index()
        veg_by_site['veg_type'] = veg_by_site.apply(classify_vegetation, axis=1)
        if 'veg_type' in df.columns:
            df = df.drop(columns=['veg_type'])
        df = df.merge(veg_by_site[['siteid', 'veg_type']], on='siteid', how='left')
    else:
        df['veg_type'] = 'unknown'
    
    # Severity classification (grass auto-classified as low)
    df['severity_class'] = df.apply(lambda r: classify_severity(r, veg_type=r.get('veg_type')), axis=1)
    
    # Size classification
    if 'area_km2' in df.columns:
        size_breaks = calculate_size_terciles(df)
        if size_breaks is not None:
            size_by_site = df.groupby('siteid')['area_km2'].first().reset_index()
            size_by_site['size_class'] = size_by_site['area_km2'].apply(lambda x: classify_fire_size(x, size_breaks))
            if 'size_class' in df.columns:
                df = df.drop(columns=['size_class'])
            df = df.merge(size_by_site[['siteid', 'size_class']], on='siteid', how='left')
            df.loc[df['treated'] == 0, 'size_class'] = 'control'
        else:
            df['size_class'] = 'unknown'
    else:
        df['size_class'] = 'unknown'
    
    return df

# Load Colorado
print("\nColorado:")
co_df = load_state_data(CO_PRED, CO_FIRE, classify_treatment_co, 'CO', co_rec_mapping)
print(f"  {len(co_df):,} observations, {co_df['siteid'].nunique()} sites")

# Load California
print("\nCalifornia:")
ca_df = load_state_data(CA_PRED, CA_FIRE, classify_treatment_ca, 'CA', ca_rec_mapping)
print(f"  {len(ca_df):,} observations, {ca_df['siteid'].nunique()} sites")

# ============================================================
# RUN HETEROGENEITY ANALYSIS
# ============================================================

print("\n" + "=" * 60)
print("ESTIMATING HETEROGENEOUS FIRE EFFECTS (SUN-ABRAHAM)")
print("=" * 60)

results = []

for state, data in [('CO', co_df), ('CA', ca_df)]:
    print(f"\n{state}:")
    
    for fire_type in ['wildfire', 'prescribed']:
        print(f"  {fire_type}:")
        
        # Size analysis
        if 'area_km2' in data.columns and data[data['treated'] == 1]['size_class'].nunique() > 1:
            for size in ['small', 'medium', 'large']:
                print(f"    {size}...", end=' ')
                
                monthly, n_t, n_c, baseline = estimate_fire_effect(
                    data, fire_type, CONTROLS, size_class=size
                )
                
                if monthly:
                    annual = aggregate_to_years(monthly)
                    if annual:
                        annual_pct = to_percent(annual, baseline)
                        for year, vals in annual_pct.items():
                            results.append({
                                'state': state,
                                'fire_type': fire_type,
                                'stratification': 'size',
                                'stratum': size,
                                'year': year,
                                'effect_pct': vals['coef'],
                                'ci_lower': vals['ci_lower'],
                                'ci_upper': vals['ci_upper'],
                                'n_treated': n_t,
                                'n_control': n_c
                            })
                        print(f"✓ ({n_t} treated)")
                    else:
                        print("insufficient data")
                else:
                    print("failed")
        else:
            print("    [Skipping size analysis - data not available]")
        
        # Severity analysis
        sev_cols = ['pct_high_severity', 'pct_moderate_severity']
        if all(c in data.columns for c in sev_cols) and data[data['treated'] == 1]['severity_class'].nunique() > 1:
            for severity in ['low', 'moderate', 'high']:
                print(f"    {severity} severity...", end=' ')
                
                monthly, n_t, n_c, baseline = estimate_fire_effect(
                    data, fire_type, CONTROLS, severity=severity
                )
                
                if monthly:
                    annual = aggregate_to_years(monthly)
                    if annual:
                        annual_pct = to_percent(annual, baseline)
                        for year, vals in annual_pct.items():
                            results.append({
                                'state': state,
                                'fire_type': fire_type,
                                'stratification': 'severity',
                                'stratum': severity,
                                'year': year,
                                'effect_pct': vals['coef'],
                                'ci_lower': vals['ci_lower'],
                                'ci_upper': vals['ci_upper'],
                                'n_treated': n_t,
                                'n_control': n_c
                            })
                        print(f"✓ ({n_t} treated)")
                    else:
                        print("insufficient data")
                else:
                    print("failed")
        else:
            print("    [Skipping severity analysis - data not available]")
        
        # Vegetation analysis
        veg_cols = ['grass_pct_mean', 'shrub_pct_mean', 'tree_pct_mean']
        if all(c in data.columns for c in veg_cols) and data[data['treated'] == 1]['veg_type'].nunique() > 1:
            veg_types = ['grass', 'tree'] if state == 'CO' else ['grass', 'shrub', 'tree']
            for veg in veg_types:
                print(f"    {veg}...", end=' ')
                
                monthly, n_t, n_c, baseline = estimate_fire_effect(
                    data, fire_type, CONTROLS, vegetation=veg
                )
                
                if monthly:
                    annual = aggregate_to_years(monthly)
                    if annual:
                        annual_pct = to_percent(annual, baseline)
                        for year, vals in annual_pct.items():
                            results.append({
                                'state': state,
                                'fire_type': fire_type,
                                'stratification': 'vegetation',
                                'stratum': veg,
                                'year': year,
                                'effect_pct': vals['coef'],
                                'ci_lower': vals['ci_lower'],
                                'ci_upper': vals['ci_upper'],
                                'n_treated': n_t,
                                'n_control': n_c
                            })
                        print(f"✓ ({n_t} treated)")
                    else:
                        print("insufficient data")
                else:
                    print("failed")
        else:
            print("    [Skipping vegetation analysis - data not available]")
        
        # Recreation County analysis
        if 'recreation_status' in data.columns and data[data['treated'] == 1]['recreation_status'].nunique() > 1:
            for rec_status in ['high_recreation', 'non_recreation']:
                rec_label = 'high-rec' if rec_status == 'high_recreation' else 'non-rec'
                print(f"    {rec_label}...", end=' ')
                
                monthly, n_t, n_c, baseline = estimate_fire_effect(
                    data, fire_type, CONTROLS, recreation_status=rec_status
                )
                
                if monthly:
                    annual = aggregate_to_years(monthly)
                    if annual:
                        annual_pct = to_percent(annual, baseline)
                        for year, vals in annual_pct.items():
                            results.append({
                                'state': state,
                                'fire_type': fire_type,
                                'stratification': 'recreation',
                                'stratum': rec_status,
                                'year': year,
                                'effect_pct': vals['coef'],
                                'ci_lower': vals['ci_lower'],
                                'ci_upper': vals['ci_upper'],
                                'n_treated': n_t,
                                'n_control': n_c
                            })
                        print(f"✓ ({n_t} treated)")
                    else:
                        print("insufficient data")
                else:
                    print("failed")
        else:
            print("    [Skipping recreation analysis - data not available]")

# ============================================================
# SAVE RESULTS
# ============================================================

print("\n" + "=" * 60)
print("RESULTS")
print("=" * 60)

if len(results) > 0:
    df_results = pd.DataFrame(results)
    df_results.to_csv(OUTPUT, index=False)
    print(f"\n✓ Saved to: {OUTPUT}")
    print(f"\nTotal estimates: {len(df_results)}")
    print(f"Stratifications: {df_results['stratification'].unique()}")
else:
    print("\nNo results generated - data may be missing required columns.")

print("\n" + "=" * 60)
print("DONE")
print("=" * 60)

if len(results) > 0:
    print(f"\nFull results saved to: {OUTPUT}")
    print("Includes all available stratifications: size, severity, vegetation, recreation")

LOADING RECREATION COUNTY DATA

Loading recreation mappings from: siteid_recreation_status.csv
  ✓ Loaded 523 siteid mappings
  CA siteid mappings: 379 unique sites
  CO siteid mappings: 144 unique sites

  Recreation status distribution:
state  recreation_status
CA     high_recreation      152
       non_recreation       227
CO     high_recreation       64
       non_recreation        80
dtype: int64

LOADING DATA

Colorado:
  Recreation status matched: 8,160/8,160 (100.0%)
  8,160 observations, 136 sites

California:
  Recreation status matched: 22,380/22,380 (100.0%)
  22,380 observations, 373 sites

ESTIMATING HETEROGENEOUS FIRE EFFECTS (SUN-ABRAHAM)

CO:
  wildfire:
    small... ✓ (12 treated)
    medium... ✓ (14 treated)
    large... ✓ (18 treated)
    low severity... ✓ (31 treated)
    moderate severity... ✓ (8 treated)
    high severity... ✓ (5 treated)
    grass... ✓ (27 treated)
    tree... ✓ (17 treated)
    high-rec... ✓ (15 treated)
    non-rec... ✓ (29 treated)
  prescrib